In [18]:
import pymupdf4llm

md = pymupdf4llm.to_markdown(r"C:\Users\thuyt\Desktop\KPDL\backend\db\papers\907-Văn bản của bài báo-3852-1-10-20210813.pdf", header=False, footer=False)
print(md)

## **Ghi nhận hình ảnh cắt lớp 3 chiều sử dụng hiệu ứng dịch chuyển pha Doppler với nguồn sáng phổ rộng** 

**Phạm Đức Quang[1, 2*] , Bành Quốc Tuấn[1] , Vũ Thanh Tùng[3] , Lê Quang Thái[2] , Nguyễn Quốc Đạt[2]** 

_1Viện Ứng dụng Công nghệ 2Trung tâm Dịch vụ Khoa học và Công nghệ, Học viện Khoa học, Công nghệ và Đổi mới sáng tạo 3Viện Cơ khí, Trường Đại học Bách khoa Hà Nội_ 

Ngày nhận bài 15/6/2020; ngày chuyển phản biện 18/6/2020; ngày nhận phản biện 12/8/2020; ngày chấp nhận đăng 18/8/2020 

## _**Tóm tăt:**_ 

**Trong nghiên cứu này, các tác giả đề xuất một phương pháp mới cho phép ghi nhận hình ảnh cắt lớp 3 chiều của mẫu vật sử dụng phương pháp giao thoa với nguồn sáng phổ rộng. Tần số của sóng tham chiếu được điều chế theo hiệu ứng Doppler do sự di chuyển với một vận tốc cố định của gương tham chiếu. Sự thay đổi cường độ ánh sáng của vân giao thoa giữa sóng tham chiếu và sóng ánh sáng phản xạ từ vật (sóng vật) do hiệu ứng Doppler tại vị trí của mỗi điểm ảnh được đặc trưng bởi 

In [19]:
import markdown
html = markdown.markdown(md)
print(html)

<h2><strong>Ghi nhận hình ảnh cắt lớp 3 chiều sử dụng hiệu ứng dịch chuyển pha Doppler với nguồn sáng phổ rộng</strong></h2>
<p><strong>Phạm Đức Quang[1, 2*] , Bành Quốc Tuấn[1] , Vũ Thanh Tùng[3] , Lê Quang Thái[2] , Nguyễn Quốc Đạt[2]</strong> </p>
<p><em>1Viện Ứng dụng Công nghệ 2Trung tâm Dịch vụ Khoa học và Công nghệ, Học viện Khoa học, Công nghệ và Đổi mới sáng tạo 3Viện Cơ khí, Trường Đại học Bách khoa Hà Nội</em> </p>
<p>Ngày nhận bài 15/6/2020; ngày chuyển phản biện 18/6/2020; ngày nhận phản biện 12/8/2020; ngày chấp nhận đăng 18/8/2020 </p>
<h2><em><strong>Tóm tăt:</strong></em></h2>
<p><strong>Trong nghiên cứu này, các tác giả đề xuất một phương pháp mới cho phép ghi nhận hình ảnh cắt lớp 3 chiều của mẫu vật sử dụng phương pháp giao thoa với nguồn sáng phổ rộng. Tần số của sóng tham chiếu được điều chế theo hiệu ứng Doppler do sự di chuyển với một vận tốc cố định của gương tham chiếu. Sự thay đổi cường độ ánh sáng của vân giao thoa giữa sóng tham chiếu và sóng ánh sáng phản 

In [22]:
import re
from bs4 import BeautifulSoup

def extract_metadata_bs4(html: str) -> dict:
    soup = BeautifulSoup(html, "lxml")  # lxml nhanh & chịu lỗi tốt hơn html.parser
    
    meta = {"title": "", "authors": "", "abstract": "", "keywords": ""}

    # 1. TITLE: Heading đầu tiên dài > 20 ký tự
    title_tag = soup.find(lambda t: t.name in ["h1", "h2"] and len(t.get_text(strip=True)) > 20)
    if title_tag:
        meta["title"] = title_tag.get_text(strip=True)

    # 2. AUTHORS: <p> chứa tên + marker [...]
    if title_tag:
        for sibling in title_tag.find_next_siblings("p"):
            text = sibling.get_text(strip=True)
            # Kiểm tra có marker [số] và ký tự tiếng Việt
            if re.search(r"\[.*?\d.*?\]", text) and re.search(r"[àáạảãăâèéẹêìíòóôơùúưýđ]", text, re.I):
                meta["authors"] = text
                break
        # Xóa marker [1], [1, 2*], [*]... và chuẩn hóa dấu phẩy
        if meta["authors"]:
            meta["authors"] = re.sub(r"\[[\d,\s*]+\]", "", meta["authors"])
            meta["authors"] = re.sub(r"\s*,\s*", ", ", meta["authors"]).strip()

    # 3. ABSTRACT: Handle typo "Tóm tăt" / "Tóm tắt" / "Abstract"
    abs_label = soup.find(lambda t: re.search(r"Tóm\s+t[ắă]t|Abstract", t.get_text(), re.I))
    if abs_label:
        # Lấy thẻ <p> đầu tiên xuất hiện SAU label (bỏ qua nesting <h2>/<em>/<strong>)
        abs_p = abs_label.find_next("p")
        if abs_p:
            meta["abstract"] = abs_p.get_text(strip=True)

    # 4. KEYWORDS: Handle "Từ khóa/Từ khoá/Keywords"
    kw_label = soup.find(lambda t: re.search(r"Từ\s+kh[oó]a|Keywords", t.get_text(), re.I))
    if kw_label:
        parent_p = kw_label.find_parent("p")
        if parent_p:
            raw_kw = parent_p.get_text(strip=True)
            # Cắt label + cắt phần "Chỉ số phân loại" nếu nằm cùng dòng
            raw_kw = re.sub(r"^Từ\s+kh[oó]a\s*:?\s*", "", raw_kw, flags=re.I)
            raw_kw = re.sub(r"\s*Chỉ\s+số\s+phân\s+loại.*$", "", raw_kw, flags=re.I)
            meta["keywords"] = raw_kw.strip().rstrip(".")
        else:
            # Fallback: nếu không nằm trong <p>, lấy <strong> kế tiếp
            kw_strong = kw_label.find_next_sibling("strong")
            if kw_strong:
                raw_kw = kw_strong.get_text(strip=True)
                raw_kw = re.sub(r"\s*Chỉ\s+số\s+phân\s+loại.*$", "", raw_kw, flags=re.I)
                meta["keywords"] = raw_kw.strip().rstrip(".")

    return meta

In [23]:
res = extract_metadata_bs4(html=html)
print(res)

{'title': 'Ghi nhận hình ảnh cắt lớp 3 chiều sử dụng hiệu ứng dịch chuyển pha Doppler với nguồn sáng phổ rộng', 'authors': 'Phạm Đức Quang, Bành Quốc Tuấn, Vũ Thanh Tùng, Lê Quang Thái, Nguyễn Quốc Đạt', 'abstract': 'Phạm Đức Quang[1, 2*] , Bành Quốc Tuấn[1] , Vũ Thanh Tùng[3] , Lê Quang Thái[2] , Nguyễn Quốc Đạt[2]', 'keywords': ''}
